###  Setup ADLS access

In [0]:
from common_scripts.common_functions import setup_adls_access

setup_adls_access(spark, dbutils)

### Read data from bronze
- pickup and transform required columns 
- drop duplicates
- create temp view to use in Incremental process

In [0]:
bronze_path = "abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/bronze/crypto/"
df_raw = spark.read.json(bronze_path)
display(df_raw)

In [0]:
from pyspark.sql.functions import col

df_clean = df_raw.select(
    col("id"),
    col("symbol"),
    col("name"),
    col("current_price").cast("double"),
    col("market_cap").cast("long"),
    col("total_volume").cast("long"),
    col("high_24h").cast("double"),
    col("low_24h").cast("double"),
    col("price_change_percentage_24h").cast("double"),
    col("ingestion_time").cast("timestamp"),
    col("ingestion_date").alias("data_load_date")
)

display(df_clean)

In [0]:
df_clean = df_clean.dropDuplicates(["id", "ingestion_time"])

In [0]:
df_clean.createOrReplaceTempView("v_crypto_bronze")

In [0]:


# silver_path = "abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto"

#  df_clean.write.format("delta") \
#     .mode("overwrite") \
#     .partitionBy("data_load_date") \
#     .save(silver_path)

- Create external tables if not done earlier
- Use MERGE statements to load data into history and latest tables

In [0]:
%sql

CREATE TABLE IF NOT EXISTS financial_project_db.silver_crypto_history
(id string,
symbol string,
name string,
current_price double,
market_cap bigint,
total_volume bigint,
high_24h double,
low_24h double,
price_change_percentage_24h double,
ingestion_time TIMESTAMP ,
data_load_date date)
USING DELTA
LOCATION 'abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto_history_partitioned';

CREATE TABLE IF NOT EXISTS financial_project_db.silver_crypto_latest
(id string,
symbol string,
name string,
current_price double,
market_cap bigint,
total_volume bigint,
high_24h double,
low_24h double,
price_change_percentage_24h double,
ingestion_time TIMESTAMP ,
data_load_date date)
USING DELTA
LOCATION 'abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/silver/crypto_lates';


In [0]:
%sql 

MERGE INTO financial_project_db.silver_crypto_history AS target
USING v_crypto_bronze AS source
ON target.id = source.id
AND target.data_load_date = source.data_load_date

WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql 

MERGE INTO financial_project_db.silver_crypto_latest AS target
USING v_crypto_bronze AS source
ON target.id = source.id 

WHEN MATCHED 
  AND target.ingestion_time < source.ingestion_time 

THEN 
  UPDATE SET *

WHEN NOT MATCHED THEN
  INSERT *

In [0]:
%sql
SELECT * FROM adb_financial_datalake.financial_project_db.silver_crypto_history;


In [0]:
%sql
SELECT * FROM adb_financial_datalake.financial_project_db.silver_crypto_latest;